# Preprocessing B — Native categorical

## Idea
**Minimal cleaning**: let XGBoost do the work that approach A does by hand.
- **Missing values**: all NaNs are kept — structural ones too ("no garage" stays NaN instead of `"None"`).
  XGBoost learns, at every split, which branch missing values should go to.
- **Categorical features**: no ordinal mapping and no one-hot encoding. Text columns are stored as pandas `category` and the model
  is trained with `enable_categorical=True`: a split can then send *any group* of categories left and the rest right,
  instead of testing one dummy column at a time.
- **Columns**: only the identifiers are dropped; mostly-empty and near-constant columns are kept.

The comparison with A answers: *does the manual cleaning and encoding help, or can XGBoost handle the raw data on its own?*

Trade-off: the order of quality ratings (`Po < Fa < TA < Gd < Ex`) is not given to the model — it has to discover useful groupings itself.

**Output:** `processed_data/B_native_categorical.csv`

## Steps

### 1. Load the data and create the target

In [1]:
import pandas as pd

from common import TARGET, add_price_class, load_processed, load_raw, save_processed

df = load_raw()
df = add_price_class(df)
print(df.shape)

(2930, 82)


### 2. Drop the identifiers
`Order` (row number) and `PID` (parcel ID) say nothing about the price. They are the only columns removed.

In [2]:
df = df.drop(columns=["Order", "PID"])
print(df.shape)

(2930, 80)


### 3. Categorical columns
`MS SubClass` is a category code stored as a number (20 = 1-story 1946+, 60 = 2-story, ...). It gets an `SC` prefix
(`20` → `SC20`) so it is read back as text from the CSV and treated as a category, not as a quantity.

In [3]:
df["MS SubClass"] = "SC" + df["MS SubClass"].astype(str)

categorical_cols = df.select_dtypes(exclude="number").columns.tolist()
print(len(categorical_cols), "categorical columns")
df[categorical_cols].nunique().sort_values(ascending=False).to_frame("categories").T

44 categorical columns


,Neighborhood,Exterior 2nd,MS SubClass,Exterior 1st,Sale Type,Condition 1,Functional,Condition 2,House Style,Roof Matl,...,Fence,Pool QC,Land Contour,Paved Drive,Utilities,Land Slope,Garage Finish,Alley,Central Air,Street
categories,28,17,16,16,10,9,8,8,8,8,...,4,4,4,3,3,3,3,2,2,2


### 4. Missing values
Nothing is filled. Structural and truly missing values both stay NaN.

In [4]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(2)})

,missing,pct
Pool QC,2917,99.56
Misc Feature,2824,96.38
Alley,2732,93.24
Fence,2358,80.48
Mas Vnr Type,1775,60.58
Fireplace Qu,1422,48.53
Lot Frontage,490,16.72
Garage Cond,159,5.43
Garage Qual,159,5.43
Garage Finish,159,5.43


## Output saved

In [5]:
assert not df.columns.str.contains(r"[\[\]<]").any(), "XGBoost does not accept [, ] or < in feature names"

path = save_processed(df, "B_native_categorical")
print(f"Saved {df.shape[0]} rows x {df.shape[1] - 1} features (+ {TARGET}) to {path.name}")

Saved 2930 rows x 79 features (+ PriceClass) to B_native_categorical.csv


A CSV file stores only text, so the `category` type is lost when saving. `load_processed()` from `common.py` converts the text
columns back to categories when the data is loaded — check that the round trip works:

In [6]:
reloaded = load_processed("B_native_categorical")
print("Categorical columns after loading:", (reloaded.dtypes == "category").sum())
print("Same shape as before saving:", reloaded.shape == df.shape)

Categorical columns after loading: 44
Same shape as before saving: True


**Summary:** 2930 houses, 79 raw features (35 numeric, 44 categorical), all missing values kept. Models must be created with
`enable_categorical=True`.